# Torch Datasets and Dataloaders


Datasets and Dataloaders are essential components in PyTorch for handling data efficiently during model training and evaluation.

### Simple Analogy: Restaurant Kitchen
- **Dataset** = Recipe book (knows how to get one dish)
- **DataLoader** = Kitchen staff (gets many dishes efficiently in batches)

**Why use them?**
``` python
# ❌ Manual approach (what you might do first)
x_data = torch.randn(10000, 4)
y_data = torch.randn(10000, 1)

# Manually create batches - tedious!
for i in range(0, len(x_data), 64):
    x_batch = x_data[i:i+64]
    y_batch = y_data[i:i+64]
    # train...
    
# ✅ With Dataset + DataLoader - automatic!
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)
for x_batch, y_batch in train_loader:
    # train... (batches created automatically!)    

## Basic Dataset Pattern

In [7]:
import torch
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self):
        # Load your data here
        self.x = torch.randn(1000, 4)  # 1000 samples, 4 features
        self.y = torch.randn(1000, 1)  # 1000 labels
    
    def __len__(self):
        return len(self.x)  # "How many samples do you have?"
    
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]  # "Give me sample #idx"


## Basic DataLoader Usage

In [ ]:
# Create dataset and loader
dataset = MyDataset()
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Training loop becomes simple
for epoch in range(10):
    for x_batch, y_batch in loader:  # Automatic batching!
        # Train your model
        pred = model(x_batch)
        loss = criterion(pred, y_batch)
        # ... rest of training

### Key Benefits for Beginners
1. ***Automatic batching*** - No manual slicing
2. ***Automatic shuffling*** - Better training
3. ***Memory efficient*** - Loads data as needed
4. ***Works with any size*** - 100 samples or 1 million

-----------------------------------------

## Iterable Dataset Example

In [25]:
import torch
from torch.utils.data import IterableDataset, DataLoader
import random

class StreamingDataset(IterableDataset):
    def __init__(self, total_samples=1000):
        self.total_samples = total_samples
    
    def __iter__(self):
        # This generates data on-the-fly as we iterate
        for i in range(self.total_samples):
            # Simulate streaming data (could be from file, network, etc.)
            x = torch.randn(4)  # 4 features
            y = torch.randn(1)  # 1 label
            yield x, y  # "yield" makes this a generator

# Usage
streaming_dataset = StreamingDataset(total_samples=128)
loader = DataLoader(streaming_dataset, batch_size=32)

# Training loop - same as before!
for epoch in range(2):
    print(f"\nEpoch {epoch + 1}")
    batch_count = 0
    for x_batch, y_batch in loader:
        batch_count += 1
        
        print(f"  Batch {batch_count}: x_batch.shape={x_batch.shape}, y_batch.shape={y_batch.shape}")
        #if batch_count >= 2:  # Just show first 2 batches
        #    break


Epoch 1
  Batch 1: x_batch.shape=torch.Size([32, 4]), y_batch.shape=torch.Size([32, 1])
  Batch 2: x_batch.shape=torch.Size([32, 4]), y_batch.shape=torch.Size([32, 1])
  Batch 3: x_batch.shape=torch.Size([32, 4]), y_batch.shape=torch.Size([32, 1])
  Batch 4: x_batch.shape=torch.Size([32, 4]), y_batch.shape=torch.Size([32, 1])

Epoch 2
  Batch 1: x_batch.shape=torch.Size([32, 4]), y_batch.shape=torch.Size([32, 1])
  Batch 2: x_batch.shape=torch.Size([32, 4]), y_batch.shape=torch.Size([32, 1])
  Batch 3: x_batch.shape=torch.Size([32, 4]), y_batch.shape=torch.Size([32, 1])
  Batch 4: x_batch.shape=torch.Size([32, 4]), y_batch.shape=torch.Size([32, 1])


### Real-World Example: Reading from File

In [ ]:
class FileStreamingDataset(IterableDataset):
    def __init__(self, file_path, chunk_size=100):
        self.file_path = file_path
        self.chunk_size = chunk_size
    
    def __iter__(self):
        # Simulate reading from a large file line by line
        # In reality, you'd open and read from an actual file
        for line_num in range(1000):  # Pretend file has 1000 lines
            # Simulate parsing a line from CSV/text file
            features = torch.randn(4)  # Parse features from line
            label = torch.tensor([line_num % 2], dtype=torch.float32)  # Binary label
            
            yield features, label

# This won't load all data into memory at once!
file_dataset = FileStreamingDataset("imaginary_large_file.csv")
loader = DataLoader(file_dataset, batch_size=64)

## Simple Vanilla DataLoader

In [1]:
import numpy as np

class VanillaDataLoader:
    def __init__(self, X, y, batch_size=32, shuffle=True):
        """
        Simple DataLoader for numpy arrays
        
        Args:
            X: numpy array of features (n_samples, n_features)
            y: numpy array of labels (n_samples,) or (n_samples, n_outputs)
            batch_size: size of each batch
            shuffle: whether to shuffle data each epoch
        """
        self.X = X
        self.y = y
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.n_samples = len(X)
        self.n_batches = (self.n_samples + batch_size - 1) // batch_size  # Ceiling division
    
    def __iter__(self):
        # Create indices for this epoch
        indices = np.arange(self.n_samples)
        
        if self.shuffle:
            np.random.shuffle(indices)
        
        # Generate batches
        for i in range(0, self.n_samples, self.batch_size):
            # Get batch indices
            batch_indices = indices[i:i + self.batch_size]
            
            # Extract batch data
            X_batch = self.X[batch_indices]
            y_batch = self.y[batch_indices]

            
            yield X_batch, y_batch
    
    def __len__(self):
        return self.n_batches

In [2]:
# Create sample numpy data
np.random.seed(42)
X_data = np.random.randn(1000, 4)  # 1000 samples, 4 features
y_data = np.random.randn(1000, 1)  # 1000 labels

print(f"Data shapes: X={X_data.shape}, y={y_data.shape}")

# Create vanilla dataloader
vanilla_loader = VanillaDataLoader(X_data, y_data, batch_size=64, shuffle=True)

print(f"Number of batches: {len(vanilla_loader)}")

# Test the dataloader
print("\nFirst few batches:")
for i, (X_batch, y_batch) in enumerate(vanilla_loader):
    print(f"Batch {i+1}: X_batch.shape={X_batch.shape}, y_batch.shape={y_batch.shape}")
    if i >= 2:  # Show first 3 batches
        break

Data shapes: X=(1000, 4), y=(1000, 1)
Number of batches: 16

First few batches:
Batch 1: X_batch.shape=(64, 4), y_batch.shape=(64, 1)
Batch 2: X_batch.shape=(64, 4), y_batch.shape=(64, 1)
Batch 3: X_batch.shape=(64, 4), y_batch.shape=(64, 1)
